# 05 · Previsões com o modelo vigente e comparação de ciclos

- Carrega `models/modelo_vigente.json` e o modelo serializado, que já traz o pré-processamento e a lista de features.
- Gera previsões para o **conjunto de teste reservado**, que nunca foi usado para escolher modelos, features ou promoção.
- Compara os ciclos (histórico local e MLflow).

In [ ]:
import sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import warnings
warnings.filterwarnings("ignore")
import json
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image

from src.utils import preparar_ambiente, carregar_config, carregar_contexto, salvar_contexto, dir_evidencias, configurar_mlflow
preparar_ambiente()
cfg = carregar_config()
pd.set_option("display.width", 160)

In [ ]:
import joblib
from src.ingestao import carregar_holdout
from src.modelagem import metricas
from src.promocao import carregar_manifesto, HISTORICO_CICLOS

ctx = carregar_contexto()
ev = dir_evidencias(ctx["cycle_id"])
manifesto = carregar_manifesto()
modelo = joblib.load(RAIZ / manifesto["arquivo_modelo"])
_, teste = carregar_holdout(cfg)
alvo = cfg["dados"]["alvo"]

pred = modelo.predict(teste.drop(columns=[alvo]))
met_teste = metricas(teste[alvo], pred)
print(f"Modelo vigente: {manifesto['algoritmo']} (promovido no ciclo {manifesto['cycle_id']}, dados {manifesto['data_version']})")
print("Features usadas:", manifesto["features"])
print(f"TESTE reservado ({len(teste)} amostras): MAE={met_teste['mae']:.3f} MPa | RMSE={met_teste['rmse']:.3f} | R²={met_teste['r2']:.3f}")

saida = teste.copy()
saida["previsao"] = pred.round(2)
saida["erro_abs"] = (saida["previsao"] - saida[alvo]).abs().round(2)
saida.to_csv(ev / "previsoes_teste.csv", index=False)
display(saida.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(teste[alvo], pred, s=14, alpha=0.7)
lim = [0, max(teste[alvo].max(), pred.max()) + 5]
ax.plot(lim, lim, "k--", lw=1); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("Real (MPa)"); ax.set_ylabel("Previsto (MPa)")
ax.set_title(f"Teste reservado – {manifesto['algoritmo']} (MAE {met_teste['mae']:.2f})")
fig.tight_layout(); fig.savefig(ev / "previsto_vs_real.png", dpi=120); plt.close(fig)
display(Image(str(ev / "previsto_vs_real.png")))

# Exemplo de uso: previsão para uma nova mistura (traço típico, 28 dias)
nova = pd.DataFrame([{"cimento": 350, "escoria": 100, "cinza_volante": 0, "agua": 180, "superplastificante": 6,
                      "agregado_graudo": 970, "agregado_miudo": 780, "idade": 28}])
print(f"Previsão para nova mistura: {modelo.predict(nova)[0]:.1f} MPa")

## Comparação entre ciclos

In [ ]:
hist = pd.read_csv(HISTORICO_CICLOS)
display(hist)

mlflow = configurar_mlflow(cfg)
runs = mlflow.search_runs(filter_string="tags.etapa = 'treino'", order_by=["start_time ASC"])
tab = runs[["tags.cycle_id", "tags.data_version", "tags.algoritmo", "metrics.val_mae", "metrics.cv_mae_media", "tags.resultado"]]
tab.columns = ["cycle_id", "data_version", "algoritmo", "val_mae", "cv_mae", "resultado"]
print("Histórico recuperado do MLflow (runs de treino):")
display(tab)

if len(hist) >= 1:
    fig, ax = plt.subplots(figsize=(8, 3.8))
    x = range(len(hist))
    ax.plot(x, hist["mae_ridge"], "o-", label="Ridge (A)")
    ax.plot(x, hist["mae_random_forest"], "s-", label="Random Forest (B)")
    for i, d in enumerate(hist["decisao"]):
        if d == "promovido":
            ax.annotate("promovido", (i, min(hist.loc[i, "mae_ridge"], hist.loc[i, "mae_random_forest"])),
                        textcoords="offset points", xytext=(0, -14), ha="center", fontsize=8, color="#1e8449")
    ax.set_xticks(list(x)); ax.set_xticklabels([f"{c}\n{l}" for c, l in zip(range(1, len(hist) + 1), hist["lote"])], fontsize=8)
    ax.set_ylabel("MAE validação fixa (MPa)"); ax.set_xlabel("ciclo / lote incorporado"); ax.legend(); ax.set_title("Evolução por ciclo")
    fig.tight_layout(); fig.savefig(RAIZ / "evidencias" / "evolucao_ciclos.png", dpi=120); plt.close(fig)
    display(Image(str(RAIZ / "evidencias" / "evolucao_ciclos.png")))

with mlflow.start_run(run_name=f"previsoes_{ctx['cycle_id']}"):
    mlflow.set_tags({"cycle_id": ctx["cycle_id"], "data_version": ctx["data_version"], "etapa": "previsoes",
                     "modelo_vigente": f"{manifesto['algoritmo']}@{manifesto['cycle_id']}"})
    mlflow.log_metrics({f"teste_{k}": v for k, v in met_teste.items()})
    mlflow.log_artifact(str(ev / "previsoes_teste.csv"), "previsoes")
salvar_contexto({"teste": met_teste})